In [ ]:
# Distance Analysis Code

# Import Required Libraries
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform as reproject_coords
from scipy.ndimage import distance_transform_edt
import os
from pathlib import Path

In [ ]:
# Set directories (update these paths to your actual file locations)
# You need a .csv file of observation GPS points in lat and long columns,
# A .csv file of landcover codes sorted by 'natural' or 'human',
# And a GeoTIFF raster file with those same landcover codes as pixel values.
# The code should automatically reproject the GPS points to match the raster's coordinate system as a safety measure

GPS_Points = r"path\to\gps_points.csv" # Update this to your GPS points .csv path
Landcover_CSV = r"path\to\landcover_codes.csv" # Update this to your landcover codes .csv path
Landcover_Raster = r"path\to\landcover.tif" # Update this to your landcover raster path
Output = r"path\to\output" # Update this to your desired output directory

In [ ]:
# Find and list input files
print("Input Files:")
print(f"GPS Points: {GPS_Points}")
print(f"Landcover CSV: {Landcover_CSV}")
print(f"Landcover Raster: {Landcover_Raster}")
print(f"Output Directory: {Output}\n")

# Load landcover types CSV
landcover_types = pd.read_csv(Landcover_CSV)
natural_codes = set(landcover_types['natural'].dropna().astype(int)) # 'natural' is a column name in the landccover CSV - update if different
human_codes = set(landcover_types['human'].dropna().astype(int)) # Same as above

print(f"Natural landcover codes: {sorted(natural_codes)}")
print(f"Human landcover codes: {sorted(human_codes)}\n")

# Load GPS points
gps_data = pd.read_csv(GPS_Points)
print(f"Loaded {len(gps_data)} GPS points")

In [ ]:
# Main analysis based on 75m buffers
# Depending on raster cell size, some points may have distances of perfect intervals if they fall within only a few pixels of a natural cover.
# For example, a raster with cell size 10m might have some points with distances of exactly 10, 20, 30, etc...
# This is normal!

# Open the raster file
with rasterio.open(Landcover_Raster) as src:
    transform = src.transform
    raster_crs = src.crs
    pixel_width = abs(transform[0])
    pixel_height = abs(transform[4])
    raster_shape = (src.height, src.width)
    nodata_value = src.nodata
    buffer_radius_m = 75.0 # Can update if needed
    radius_rows = int(np.ceil(buffer_radius_m / pixel_height))
    radius_cols = int(np.ceil(buffer_radius_m / pixel_width))

    print(f"Raster CRS: {raster_crs}")
    print(f"Raster size: {raster_shape[1]} x {raster_shape[0]} pixels")
    print(f"Pixel size: {pixel_width} x {pixel_height} meters")
    print(f"Landcover classification buffer: {buffer_radius_m:.0f} meters\n")

    # Convert to real-world distance (in meters for EPSG:3087)
    avg_pixel_size = (pixel_width + pixel_height) / 2

    # Process each GPS point 
    print("Processing GPS points...")
    results = []

    for idx, row in gps_data.iterrows():
        lat = row['decimalLatitude'] # Latitude column name in the GPS CSV - update if different
        lon = row['decimalLongitude'] # Longitude column name in the GPS CSV - update if different

        # Reproject GPS coordinates from WGS84 to raster CRS
        xs, ys = reproject_coords('EPSG:4326', raster_crs, [lon], [lat])
        x_proj, y_proj = xs[0], ys[0]

        # Convert reprojected coordinates to pixel row/col
        py, px = rowcol(transform, x_proj, y_proj)

        # Check if point is within raster bounds
        if 0 <= py < raster_shape[0] and 0 <= px < raster_shape[1]:
            # Read a window large enough to cover the 75 m buffer
            row_min = max(0, py - radius_rows)
            row_max = min(raster_shape[0], py + radius_rows + 1)
            col_min = max(0, px - radius_cols)
            col_max = min(raster_shape[1], px + radius_cols + 1)

            window = rasterio.windows.Window(
                col_min,
                row_min,
                col_max - col_min,
                row_max - row_min
            )
            window_data = src.read(1, window=window)

            # Determine dominant landcover within a 75 m circular buffer
            row_offsets_m = (np.arange(row_min, row_max) - py) * pixel_height
            col_offsets_m = (np.arange(col_min, col_max) - px) * pixel_width
            buffer_mask = (
                row_offsets_m[:, None] ** 2 + col_offsets_m[None, :] ** 2
            ) <= buffer_radius_m ** 2

            valid_buffer_mask = buffer_mask.copy()
            if nodata_value is not None:
                valid_buffer_mask &= window_data != nodata_value
            if np.issubdtype(window_data.dtype, np.floating):
                valid_buffer_mask &= ~np.isnan(window_data)

            buffer_values = window_data[valid_buffer_mask]
            if buffer_values.size > 0:
                unique_values, counts = np.unique(buffer_values.astype(int), return_counts=True)
                point_landcover = int(unique_values[np.argmax(counts)])
            else:
                point_landcover = np.nan

            # Use the raster cell at the GPS point itself for distance-to-natural calculations
            py_local = py - row_min
            px_local = px - col_min
            center_landcover = window_data[py_local, px_local]
            if nodata_value is not None and center_landcover == nodata_value:
                center_landcover = np.nan
            elif np.issubdtype(window_data.dtype, np.floating) and np.isnan(center_landcover):
                center_landcover = np.nan
            else:
                center_landcover = int(center_landcover)

            # Check if point is already on natural landcover
            if center_landcover in natural_codes:
                nearest_natural = center_landcover
                distance_real = 0.0
            else:
                found = False
                search_radius = 10  # Start with 10 pixel radius
                max_search_radius = 10000  # Maximum search radius in pixels (100 km)
                nearest_natural = np.nan
                min_distance = np.inf

                while not found and search_radius <= max_search_radius:
                    row_min = max(0, py - search_radius)
                    row_max = min(raster_shape[0], py + search_radius + 1)
                    col_min = max(0, px - search_radius)
                    col_max = min(raster_shape[1], px + search_radius + 1)

                    window = rasterio.windows.Window(
                        col_min,
                        row_min,
                        col_max - col_min,
                        row_max - row_min
                    )
                    search_window = src.read(1, window=window)

                    window_natural_mask = np.isin(search_window, list(natural_codes))
                    if nodata_value is not None:
                        window_natural_mask &= search_window != nodata_value
                    if np.issubdtype(search_window.dtype, np.floating):
                        window_natural_mask &= ~np.isnan(search_window)

                    if np.any(window_natural_mask):
                        y_coords, x_coords = np.where(window_natural_mask)
                        py_local = py - row_min
                        px_local = px - col_min
                        distances = np.sqrt((y_coords - py_local) ** 2 + (x_coords - px_local) ** 2)
                        nearest_idx = np.argmin(distances)
                        min_distance = distances[nearest_idx]
                        nearest_natural = int(search_window[y_coords[nearest_idx], x_coords[nearest_idx]])
                        found = True
                    else:
                        search_radius *= 2

                distance_real = min_distance * avg_pixel_size

            results.append({
                'Lat': lat,
                'Long': lon,
                'Point_Landcover': point_landcover,
                'Nearest_Natural': nearest_natural,
                'Distance_meters': distance_real
            })
        else:
            print(f"Warning: GPS point ({lat}, {lon}) is outside raster bounds")

        if (idx + 1) % 100 == 0 or (idx + 1) == len(gps_data):
            print(f"  Processed {idx + 1}/{len(gps_data)} GPS points")

    all_points_df = pd.DataFrame(results)
    human_points_df = all_points_df[all_points_df['Point_Landcover'].isin(human_codes)].copy()

    print(f"\nTotal points processed: {len(all_points_df)}")
    print(f"Points in human landcover: {len(human_points_df)}")

    if len(all_points_df) > 0:
        print(f"\nDistance Statistics (meters):")
        print(f"  Mean: {all_points_df['Distance_meters'].mean():.2f}")
        print(f"  Median: {all_points_df['Distance_meters'].median():.2f}")
        print(f"  Min: {all_points_df['Distance_meters'].min():.2f}")
        print(f"  Max: {all_points_df['Distance_meters'].max():.2f}")
        print(f"  Points with distance = 0: {(all_points_df['Distance_meters'] == 0).sum()}")
        print(f"  Points with distance <= 10m: {(all_points_df['Distance_meters'] <= 10).sum()}")
        print(f"  Points with distance <= 100m: {(all_points_df['Distance_meters'] <= 100).sum()}")

    all_points_path = os.path.join(Output, 'all_points.csv')
    human_points_path = os.path.join(Output, 'human_use_points.csv')

    all_points_df.to_csv(all_points_path, index=False)
    human_points_df.to_csv(human_points_path, index=False)

    print(f"\nOutput files saved:")
    print(f"- {all_points_path}")
    print(f"- {human_points_path}")